# English → Dutch Translation — Inference Only

Runs the **final trained checkpoint** (epoch 17, word-level tokenizer) to translate English sentences into Dutch. No training happens here — this notebook just downloads the finished model from Google Drive and runs it.

- Tokenizer: **word-level** (the project default — this is the version used in the project report, not the BPE variant)
- Checkpoint: `tmodel_17.pt` (best validation BLEU-4: 24.21)
- Source code: [transformers-from-scratch](https://github.com/abhijitdalal26/transformers-from-scratch)

**To use this notebook:** Runtime → Run all. Then edit the sentence in the last cell and re-run it as many times as you like.

## 1. Get the code

In [ ]:
!git clone --depth 1 https://github.com/abhijitdalal26/transformers-from-scratch.git
%cd transformers-from-scratch/pytorch-transformer

In [ ]:
!pip install -q gdown tokenizers datasets

## 2. Download the final model from Google Drive

This pulls the checkpoint + word-level tokenizers (`tmodel_17.pt`, `tokenizer_en.json`, `tokenizer_nl.json`) from the shared Drive folder. No sign-in needed — the folder is shared with "anyone with the link".

In [ ]:
import os, glob

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1_05qFOIca1fdhYmUpTHRVugH2YeYSdpq?usp=sharing"
WEIGHTS_DIR = "/content/weights"

os.makedirs(WEIGHTS_DIR, exist_ok=True)
!gdown --folder "$DRIVE_FOLDER_URL" -O "$WEIGHTS_DIR" --remaining-ok

# gdown nests the files one level down (inside a folder named after the Drive
# folder), so locate the actual files instead of assuming a fixed path.
def find_one(pattern):
    hits = glob.glob(os.path.join(WEIGHTS_DIR, "**", pattern), recursive=True)
    if not hits:
        raise FileNotFoundError(f"Could not find {pattern} under {WEIGHTS_DIR} — check the Drive folder contents/sharing.")
    return hits[0]

checkpoint_path = find_one("tmodel_17.pt")
model_folder = os.path.dirname(checkpoint_path)
tokenizer_en = find_one("tokenizer_en.json")

print("Model folder:", model_folder)
print("Checkpoint:", checkpoint_path)
print("Tokenizer dir:", os.path.dirname(tokenizer_en))

## 3. Point the project config at the downloaded weights

`config.py` already defaults to the word-level tokenizer (`tokenizer_type="wordlevel"`). `MODEL_FOLDER` / `TOKENIZER_FILE` just redirect it from the local `model/` folder to the files we just downloaded — no code changes.

In [ ]:
import os

os.environ["MODEL_FOLDER"] = model_folder
os.environ["TOKENIZER_FILE"] = os.path.join(os.path.dirname(tokenizer_en), "tokenizer_{0}.json")
# TOKENIZER_TYPE left unset -> defaults to "wordlevel" (see config.py)

import config as cfgmod
from config import resolve_checkpoint
from translate import translate

config = cfgmod.get_config()
checkpoint = resolve_checkpoint(config, "17")
print("Using checkpoint:", checkpoint)

## 4. Translate

Edit `sentence` below and re-run this cell for any new input.

In [ ]:
sentence = "Where is the station?"
_ = translate(sentence, config=config, checkpoint=checkpoint)

### A few more examples

In [ ]:
for s in [
    "We were very happy yesterday.",
    "I have two brothers and one sister.",
    "The man who came yesterday is my friend.",
]:
    translate(s, config=config, checkpoint=checkpoint)
    print()